# Exploratory data analysis

Overview of one CARE to Compare wind farm, plus an interactive plot of selected turbines and features.

Pick a farm in the dropdown below, then run the cells below it (in VS Code: **Run Below** / *Execute Cell and Below*). Changing the farm later needs another *Run Below*. Only farms already converted with `python data_process.py --farms <farm>` are listed.

In [267]:
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import ipywidgets as widgets
from IPython.display import display

from loaders import DATA_DIR, FARMS, META_COLUMNS, NORMAL_STATUS, load_event, load_event_info, load_feature_description

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 80)

available_farms = [f for f in FARMS if (DATA_DIR / f / "event_info.parquet").exists()]
farm_select = widgets.Dropdown(options=available_farms, description="Farm")
display(farm_select)

Dropdown(description='Farm', options=('A', 'C'), value='A')

In [268]:
FARM = farm_select.value
print(f"Using farm {FARM}")

Using farm A


## Load metadata

Each dataset file is one turbine's history around one event, so a turbine can appear in several files. Only the `asset_id` and `time_stamp` columns are read here, to keep this fast for Farm C.

In [269]:
event_info = load_event_info(FARM).rename(columns={"asset": "asset_id"})  # Farm A/B call it `asset`
feature_desc = load_feature_description(FARM)
# The source CSVs lost the degree sign (stored as U+FFFD); restore it for display.
feature_desc["unit"] = feature_desc["unit"].str.replace("�", "°")

dataset_files = sorted((DATA_DIR / FARM / "datasets").glob("*.parquet"), key=lambda p: int(p.stem))
rows = []
for path in dataset_files:
    t = pq.read_table(path, columns=["asset_id", "time_stamp"]).to_pandas()
    rows.append({
        "event_id": int(path.stem),
        "asset_id": int(t["asset_id"].iloc[0]),
        "n_rows": len(t),
        "start": t["time_stamp"].min(),
        "end": t["time_stamp"].max(),
    })
datasets = pd.DataFrame(rows)
datasets

,event_id,asset_id,n_rows,start,end
0,0,0,54986,2022-08-04 06:10:00,2023-08-24 06:10:00
1,3,10,55487,2022-04-27 03:00:00,2023-05-20 01:10:00
2,10,10,53592,2022-10-09 08:40:00,2023-10-18 08:40:00
3,13,21,54010,2022-04-30 13:20:00,2023-05-25 10:20:00
4,14,13,54197,2022-03-03 14:00:00,2023-03-16 18:40:00
5,17,10,55090,2022-10-31 15:20:00,2023-11-20 00:40:00
6,22,21,53036,2022-08-12 09:50:00,2023-08-20 09:50:00
7,24,0,55003,2022-04-24 15:00:00,2023-05-13 11:20:00
8,25,11,54712,2022-05-23 06:50:00,2023-06-09 02:30:00
9,26,0,53702,2022-10-12 10:20:00,2023-10-22 10:20:00


## Turbines

In [270]:
turbine_ids = sorted(datasets["asset_id"].unique().tolist())
print(f"Farm {FARM}: {len(turbine_ids)} unique turbine IDs across {len(datasets)} datasets")
datasets.groupby("asset_id").agg(n_datasets=("event_id", "size"), event_ids=("event_id", list))

Farm A: 5 unique turbine IDs across 22 datasets


,n_datasets,event_ids
asset_id,,
0,5,"[0, 24, 26, 71, 73]"
10,5,"[3, 10, 17, 40, 42]"
11,4,"[25, 68, 69, 92]"
13,4,"[14, 38, 45, 84]"
21,4,"[13, 22, 51, 72]"


## Features

Sensor columns are named `<sensor_name>_<stat>` (e.g. `sensor_0_avg`); description and unit come from `feature_description`.

In [271]:
all_columns = pq.read_schema(dataset_files[0]).names
feature_cols = [c for c in all_columns if c not in META_COLUMNS]

def sensor_name(col):
    return col.rsplit("_", 1)[0] if col.rsplit("_", 1)[-1] in ("avg", "min", "max", "std") else col

desc = feature_desc.set_index("sensor_name")
features = pd.DataFrame({"column": feature_cols})
features["sensor_name"] = features["column"].map(sensor_name)
features = features.join(desc[["description", "unit"]], on="sensor_name")
print(f"{len(feature_cols)} feature columns (+ meta columns {list(META_COLUMNS)})")
features

81 feature columns (+ meta columns ['time_stamp', 'asset_id', 'id', 'train_test', 'status_type_id'])


,column,sensor_name,description,unit
0,sensor_0_avg,sensor_0,Ambient temperature,°C
1,sensor_1_avg,sensor_1,Wind absolute direction,°
2,sensor_2_avg,sensor_2,Wind relative direction,°
3,wind_speed_3_avg,wind_speed_3,Windspeed,m/s
4,wind_speed_4_avg,wind_speed_4,Estimated windspeed,m/s
5,wind_speed_3_max,wind_speed_3,Windspeed,m/s
6,wind_speed_3_min,wind_speed_3,Windspeed,m/s
7,wind_speed_3_std,wind_speed_3,Windspeed,m/s
8,sensor_5_avg,sensor_5,Pitch angle,°
9,sensor_5_max,sensor_5,Pitch angle,°


## Event descriptions

In [272]:
(event_info.groupby(["event_description", "event_label"], observed=True, dropna=False)
    .agg(n_events=("event_id", "size"), turbines=("asset_id", lambda s: sorted(s.unique())))
    .reset_index()
    .sort_values(["event_label", "n_events"], ascending=[True, False]))

,event_description,event_label,n_events,turbines
3,Hydraulic group,anomaly,6,"[0, 10, 13, 21]"
1,Gearbox failure,anomaly,2,"[10, 21]"
2,Generator bearing failure,anomaly,2,"[0, 10]"
0,Gearbox bearings damaged,anomaly,1,[21]
4,Transformer failure,anomaly,1,[11]
5,NaN,normal,10,"[0, 10, 11, 13, 21]"


## Status codes

`status_type_id` is an integer in every row. The data files don't include descriptions; the meanings below come from the CARE to Compare paper (Gück et al., [arXiv:2404.10320](https://arxiv.org/abs/2404.10320), Section 3.3). For farms B and C the codes are derived from the operators' real operating modes. Farm A had no operating-mode data, so its codes were built from EDP's fault logbook: it only flags faults, and the 14 days before each logged fault are marked as status 4. Rows per code are counted across all of this farm's datasets, split into the train and prediction periods (overlapping datasets mean some timestamps are counted more than once).

In [273]:
STATUS_DESCRIPTIONS = {  # from the CARE to Compare paper, Table "Status-ID"
    0: "Normal operation without limitations",
    1: "Derated power generation with a power restriction",
    2: "Asset is idling and waits to operate again",
    3: "Asset is in service mode / service team is at the site",
    4: "Asset is down due to a fault or other reasons",
    5: "Other operational states, e.g. system test, setup, ice build-up or emergency power",
}

status = pd.concat(
    pq.read_table(path, columns=["status_type_id", "train_test"]).to_pandas() for path in dataset_files)
status_counts = status.groupby(["status_type_id", "train_test"]).size().unstack(fill_value=0)
status_counts["total"] = status_counts.sum(axis=1)
status_counts["share"] = (status_counts["total"] / status_counts["total"].sum()).map("{:.1%}".format)
status_counts.insert(0, "considered_normal", status_counts.index.isin(NORMAL_STATUS))
status_counts.insert(0, "description", status_counts.index.map(STATUS_DESCRIPTIONS))
status_counts

train_test,description,considered_normal,prediction,train,total,share
status_type_id,,,,,,
0,Normal operation without limitations,True,29355,1024381,1053736,88.1%
3,Asset is in service mode / service team is at the site,False,3220,20469,23689,2.0%
4,Asset is down due to a fault or other reasons,False,18018,101304,119322,10.0%


## Time range

In [274]:
earliest, latest = datasets["start"].min(), datasets["end"].max()
print(f"Earliest timestamp: {earliest}")
print(f"Latest timestamp:   {latest}")
print(f"Span: {latest - earliest}")

Earliest timestamp: 2022-01-01 00:00:00
Latest timestamp:   2023-11-20 00:40:00
Span: 688 days 00:40:00


## Interactive plot

Tick the turbines and features to plot, then press **Plot**. Use the filter box to search features by column name or description; ticked features stay selected while you change the filter. Fault windows of the ticked failure modes are shaded in the turbine's colour (untick all for no shading). Each feature gets its own panel with one line per turbine. A turbine's datasets are concatenated, with overlapping timestamps kept once. Up to 8 turbines can be plotted at a time so each one keeps a distinct colour.

In [275]:
# Categorical palette in fixed order (colour-blind-checked); a turbine's colour is set by its position in the selection.
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
MAX_TURBINES = len(PALETTE)

def load_turbine(asset_id, columns):
    """All rows for one turbine across its datasets, deduplicated on timestamp."""
    frames = [load_event(FARM, eid, columns=columns, add_labels=False)
              for eid in datasets.loc[datasets["asset_id"] == asset_id, "event_id"]]
    df = pd.concat(frames).drop_duplicates("time_stamp").sort_values("time_stamp")
    return df.set_index("time_stamp")[columns]

def fault_windows(asset_id, modes):
    """(start, end, description) of this turbine's anomaly events whose description is in `modes`."""
    ev = event_info[(event_info["asset_id"] == asset_id) & (event_info["event_label"] == "anomaly")
                    & event_info["event_description"].isin(modes)]
    return list(zip(ev["event_start"], ev["event_end"], ev["event_description"]))

def plot_selection(turbines, columns, resample, modes):
    fig, axes = plt.subplots(len(columns), 1, figsize=(13, 2.6 * len(columns)), sharex=True, squeeze=False)
    axes = axes[:, 0]
    for i, asset_id in enumerate(turbines):
        color = PALETTE[i]
        df = load_turbine(asset_id, columns)
        if resample != "raw":
            df = df.resample(resample).mean()
        for ax, col in zip(axes, columns):
            ax.plot(df.index, df[col], color=color, lw=1, label=f"Turbine {asset_id}")
            for start, end, _ in fault_windows(asset_id, modes):
                ax.axvspan(start, end, color=color, alpha=0.12, lw=0)
    unit_of = features.set_index("column")["unit"]
    desc_of = features.set_index("column")["description"]
    for ax, col in zip(axes, columns):
        title = col if pd.isna(desc_of.get(col)) else f"{col} - {desc_of[col]}"
        ax.set_title(title, loc="left", fontsize=10, color="#0b0b0b")
        unit = unit_of.get(col)
        ax.set_ylabel("" if pd.isna(unit) else unit, color="#52514e")
        ax.grid(True, color="#e6e5e0", lw=0.6)
        ax.set_axisbelow(True)
        for side in ("top", "right"):
            ax.spines[side].set_visible(False)
        for side in ("left", "bottom"):
            ax.spines[side].set_color("#b5b4ad")
        ax.tick_params(colors="#52514e", labelsize=9)
    axes[-1].xaxis.set_major_formatter(mdates.ConciseDateFormatter(axes[-1].xaxis.get_major_locator()))
    if len(turbines) > 1:
        axes[0].legend(loc="upper left", bbox_to_anchor=(1.01, 1), frameon=False, fontsize=9)
    fig.tight_layout()
    plt.close(fig)  # stop matplotlib displaying it itself; the caller displays it once
    return fig

In [276]:
# Selection widgets shared by both interactive plots. Checkboxes and plain dropdowns are used rather than
# multi-select lists (need Cmd-click, which VS Code's notebook view doesn't pass through) or comboboxes
# (only suggest options matching the text already typed).
from functools import partial

feature_options = [(c if pd.isna(d) else f"{c}  ({d})", c) for c, d in zip(features["column"], features["description"])]
label_to_col = dict(feature_options)
col_to_label = {col: label for label, col in feature_options}


def turbine_checklist(n_default=1):
    """Grid of turbine checkboxes. Returns (widget, get_selected)."""
    boxes = {t: widgets.Checkbox(value=i < n_default, description=f"Turbine {t}", indent=False,
                                 layout=widgets.Layout(width="115px"))
             for i, t in enumerate(turbine_ids)}
    def set_all(value):
        for b in boxes.values():
            b.value = value
    none_btn = widgets.Button(description="Clear", layout=widgets.Layout(width="70px"))
    none_btn.on_click(lambda _: set_all(False))
    grid = widgets.GridBox(list(boxes.values()), layout=widgets.Layout(
        grid_template_columns="repeat(3, 120px)", max_height="230px", overflow="auto"))
    widget = widgets.VBox([widgets.HTML("<b>Turbines</b>"), grid, none_btn])
    return widget, lambda: [t for t, b in boxes.items() if b.value]


class FeatureChecklist:
    """Filterable checkbox list of features; ticked features persist across filter changes."""

    def __init__(self, default, max_shown=150):
        self.selected = list(default)
        self.max_shown = max_shown
        self.filter = widgets.Text(placeholder="search name or description, e.g. temp",
                                   description="Filter", layout=widgets.Layout(width="480px"))
        self.box = widgets.VBox(layout=widgets.Layout(width="480px", max_height="260px", overflow="auto",
                                                      border="1px solid #d6d5d0"))
        self.summary = widgets.HTML(layout=widgets.Layout(width="480px"))
        self.filter.observe(lambda _: self._render(), names="value")
        self._render()
        self.widget = widgets.VBox([widgets.HTML("<b>Features</b>"), self.filter, self.box, self.summary])

    def _toggle(self, col, change):
        if change["new"] and col not in self.selected:
            self.selected.append(col)
        elif not change["new"] and col in self.selected:
            self.selected.remove(col)
        self._update_summary()

    def _render(self):
        query = self.filter.value.lower()
        matches = [(label, col) for label, col in feature_options if query in label.lower()]
        boxes = []
        for label, col in matches[: self.max_shown]:
            cb = widgets.Checkbox(value=col in self.selected, description=label, indent=False,
                                  layout=widgets.Layout(width="auto"))
            cb.observe(partial(self._toggle, col), names="value")
            boxes.append(cb)
        if len(matches) > self.max_shown:
            boxes.append(widgets.HTML(f"<i>{len(matches) - self.max_shown} more; refine the filter</i>"))
        if not matches:
            boxes.append(widgets.HTML("<i>No matching features</i>"))
        self.box.children = boxes
        self._update_summary()

    def _update_summary(self):
        self.summary.value = "Selected: " + (", ".join(self.selected) if self.selected else "<i>none</i>")


def feature_dropdown(description, default):
    """Dropdown of all features with a filter box beside it. Returns (widget, dropdown)."""
    dd = widgets.Dropdown(options=feature_options, value=default, description=description,
                          layout=widgets.Layout(width="460px"))
    filt = widgets.Text(placeholder="filter", layout=widgets.Layout(width="160px"))
    def on_filter(change):
        query, current = change["new"].lower(), dd.value
        opts = [o for o in feature_options if query in o[0].lower()]
        if current not in [c for _, c in opts]:
            opts = [(col_to_label[current], current)] + opts  # keep the current choice so it doesn't jump
        dd.options = opts
        dd.value = current
    filt.observe(on_filter, names="value")
    return widgets.HBox([dd, filt]), dd


import time

# VS Code's notebook view can deliver one button click several times. The repeats queue up behind the
# first plot and run straight after it, so clicks arriving within this many seconds of the last plot
# finishing are ignored.
DUPLICATE_CLICK_WINDOW = 0.5
_last_render_done = {}


def render_plot(out, button, make_fig):
    """Replace `out`'s contents with the figure from make_fig() (or the message it raises as ValueError)."""
    if time.monotonic() - _last_render_done.get(id(button), float("-inf")) < DUPLICATE_CLICK_WINDOW:
        return
    try:
        _render_plot(out, button, make_fig)
    finally:
        _last_render_done[id(button)] = time.monotonic()


def set_output(out, *objs, text=None):
    """Replace everything in Output widget `out` with `text` and/or the rich display of each object.

    This sets the widget's `outputs` state directly instead of using clear_output() and `with out:`.
    VS Code doesn't reliably apply clears sent from widget callbacks, so those made outputs pile up.
    """
    fmt = get_ipython().display_formatter.format
    outputs = [] if text is None else [{"output_type": "stream", "name": "stdout", "text": text}]
    for obj in objs:
        data, metadata = fmt(obj)
        outputs.append({"output_type": "display_data", "data": data, "metadata": metadata})
    out.outputs = tuple(outputs)


def _render_plot(out, button, make_fig):
    import traceback
    button.disabled = True
    set_output(out, text="Loading...\n")
    try:
        fig = make_fig()
    except ValueError as err:
        set_output(out, text=f"{err}\n")
        return
    except Exception:
        set_output(out, text=traceback.format_exc())
        return
    finally:
        button.disabled = False
    set_output(out, *(fig if isinstance(fig, tuple) else (fig,)))


def check_selection(turbines, columns=True):
    if not turbines or not columns:
        raise ValueError("Select at least one turbine and one feature.")
    if len(turbines) > MAX_TURBINES:
        raise ValueError(f"Select at most {MAX_TURBINES} turbines (got {len(turbines)}).")


def failure_mode_checklist(max_label=70):
    """Checkbox per anomaly event description (all ticked). Returns (widget, get_selected)."""
    anomalies = event_info[event_info["event_label"] == "anomaly"]
    counts = anomalies.groupby("event_description")["asset_id"].agg(["size", lambda s: sorted(s.unique())])
    counts.columns = ["n", "turbines"]
    boxes = {}
    for desc, row in counts.sort_values("n", ascending=False).iterrows():
        short = desc if len(desc) <= max_label else desc[: max_label - 1] + "…"
        turbines = ", ".join(map(str, row["turbines"]))
        boxes[desc] = widgets.Checkbox(
            value=True, indent=False, layout=widgets.Layout(width="auto"),
            description=f"{short}  ({row['n']} event{'s' if row['n'] > 1 else ''}; turbine {turbines})",
            tooltip=desc)
    def set_all(value):
        for b in boxes.values():
            b.value = value
    all_btn = widgets.Button(description="Select all", layout=widgets.Layout(width="90px"))
    none_btn = widgets.Button(description="Clear", layout=widgets.Layout(width="70px"))
    all_btn.on_click(lambda _: set_all(True))
    none_btn.on_click(lambda _: set_all(False))
    box = widgets.VBox(list(boxes.values()), layout=widgets.Layout(
        max_height="200px", overflow="auto", border="1px solid #d6d5d0", width="760px"))
    widget = widgets.VBox([
        widgets.HTML("<b>Failure modes</b> (fault windows to show; hover a label for the full text)"),
        box, widgets.HBox([all_btn, none_btn])])
    return widget, lambda: [d for d, b in boxes.items() if b.value]

In [277]:
turbine_widget, selected_turbines = turbine_checklist()
feature_list = FeatureChecklist(default=feature_cols[:1])
resample_select = widgets.Dropdown(
    options=[("Raw (10 min)", "raw"), ("Hourly mean", "1h"), ("Daily mean", "1D")], value="1h",
    description="Resample")
mode_widget, selected_modes = failure_mode_checklist()
plot_button = widgets.Button(description="Plot", button_style="primary")
out = widgets.Output()

def make_timeseries():
    turbines, columns = selected_turbines(), list(feature_list.selected)
    check_selection(turbines, columns)
    return plot_selection(turbines, columns, resample_select.value, selected_modes())

def on_plot(_):
    render_plot(out, plot_button, make_timeseries)

plot_button.on_click(on_plot)

display(widgets.VBox([
    widgets.HBox([turbine_widget, feature_list.widget], layout=widgets.Layout(gap="24px")),
    mode_widget,
    widgets.HBox([resample_select, plot_button]),
    out,
]))

## Feature vs feature

Scatter one feature against another for the selected turbines, e.g. power against wind speed for a power curve. Type in the box beside X or Y to narrow its dropdown by column name or description. Points inside fault windows of the ticked failure modes are drawn in black on top, to see how faults sit relative to normal behaviour; untick all to show every point in its turbine's colour.

In [278]:
def load_turbine_rows(asset_id, columns, modes):
    """All rows for one turbine (selected columns + `is_normal_status`), deduplicated on timestamp.

    `in_fault` marks rows inside a fault window of one of the selected failure modes. It is taken from the
    event table's timestamps rather than each dataset's own `is_anomaly`, because a turbine's datasets
    overlap and deduplication could otherwise keep an unlabelled copy of a fault-window row.
    """
    frames = [load_event(FARM, eid, columns=columns)
              for eid in datasets.loc[datasets["asset_id"] == asset_id, "event_id"]]
    df = pd.concat(frames).drop_duplicates("time_stamp").sort_values("time_stamp")
    df["in_fault"] = False
    for start, end, _ in fault_windows(asset_id, modes):
        df.loc[df["time_stamp"].between(start, end), "in_fault"] = True
    return df

def default_col(prefix, prefer=None):
    cands = features[features["column"].str.startswith(prefix) & features["column"].str.endswith("_avg")]
    if prefer is not None:
        hit = cands[cands["description"].str.contains(prefer, case=False, na=False)]
        cands = hit if len(hit) else cands
    return cands["column"].iloc[0] if len(cands) else feature_cols[0]

def plot_scatter(turbines, x_col, y_col, normal_only, modes, max_points):
    fig, ax = plt.subplots(figsize=(9, 6.5))
    fault_parts = []
    for i, asset_id in enumerate(turbines):
        df = load_turbine_rows(asset_id, list(dict.fromkeys([x_col, y_col])), modes).dropna(subset=[x_col, y_col])
        if normal_only:
            df = df[df["is_normal_status"]]
        faults = df[df["in_fault"]]
        normal = df[~df["in_fault"]]
        if len(normal) > max_points:
            normal = normal.sample(max_points, random_state=0)
        ax.scatter(normal[x_col], normal[y_col], s=3, alpha=0.25, color=PALETTE[i], lw=0, rasterized=True)
        ax.scatter([], [], s=30, color=PALETTE[i], label=f"Turbine {asset_id}")  # opaque legend marker
        fault_parts.append(faults)
    faults = pd.concat(fault_parts)
    if len(faults):
        ax.scatter(faults[x_col], faults[y_col], s=4, alpha=0.5, color="#0b0b0b", lw=0, rasterized=True)
        ax.scatter([], [], s=30, color="#0b0b0b", label=f"Fault window ({len(faults):,} pts)")

    unit_of = features.set_index("column")["unit"]
    def axis_label(col):
        unit = unit_of.get(col)
        return col_to_label[col] + ("" if pd.isna(unit) else f" [{unit}]")
    ax.set_xlabel(axis_label(x_col), color="#52514e")
    ax.set_ylabel(axis_label(y_col), color="#52514e")
    ax.grid(True, color="#e6e5e0", lw=0.6)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color("#b5b4ad")
    ax.tick_params(colors="#52514e", labelsize=9)
    ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), frameon=False, fontsize=9)
    fig.tight_layout()
    plt.close(fig)  # stop matplotlib displaying it itself; the caller displays it once
    return fig

In [279]:
sc_turbine_widget, sc_selected_turbines = turbine_checklist()
sc_x_widget, sc_x = feature_dropdown("X", default_col("wind_speed"))
sc_y_widget, sc_y = feature_dropdown("Y", default_col("power", prefer="grid power"))
sc_normal = widgets.Checkbox(value=False, description="Normal status only (0, 2)")
sc_mode_widget, sc_selected_modes = failure_mode_checklist()
sc_points = widgets.Dropdown(options=[("20k pts / turbine", 20_000), ("50k pts / turbine", 50_000),
                                      ("All points", 10**9)], value=20_000, description="Sample")
sc_button = widgets.Button(description="Plot", button_style="primary")
sc_out = widgets.Output()

def make_scatter():
    turbines = sc_selected_turbines()
    check_selection(turbines)
    return plot_scatter(turbines, sc_x.value, sc_y.value, sc_normal.value, sc_selected_modes(), sc_points.value)

def on_scatter(_):
    render_plot(sc_out, sc_button, make_scatter)

sc_button.on_click(on_scatter)
display(widgets.VBox([
    widgets.HBox([sc_turbine_widget,
                  widgets.VBox([sc_x_widget, sc_y_widget, sc_normal, sc_points])],
                 layout=widgets.Layout(gap="24px")),
    sc_mode_widget,
    sc_button,
    sc_out,
]))

## Data cleaning: McKinnon et al. (2022)

McKinnon, C., Tartt, K., Carroll, J., McDonald, A., Plumley, C. & Ferguson, D. (2022). *Comparison of novel SCADA data cleaning technique for wind turbine electric pitch system.* Journal of Physics: Conference Series 2151, 012005. [doi:10.1088/1742-6596/2151/1/012005](https://doi.org/10.1088/1742-6596/2151/1/012005) ([full text](https://strathprints.strath.ac.uk/79195/))

The method first drops physically implausible power readings, then splits the power curve into roughly 1 m/s wind-speed bins and measures each record's Mahalanobis distance from its bin's mean (wind speed, power). Records further than 2 standard deviations of that distance are removed, which takes out curtailment, derating and other off-curve behaviour while keeping most of the normal power curve.

Quotes below are from Section 3.2 of the paper. Before the paper's steps we add one step of our own (step 0, removing missing values).

All functions return a boolean mask of records to remove, and `mckinnon_clean` combines them into one `removed_by` label per record. That way the before/after plot can show which step removed what.

### Configuration

Pick a farm and turbine and set the cleaning parameters on the left; the results appear on the right once the last cell of this section has run (run the whole section with **Run Below**). The widgets keep the `CLEAN_CFG` dict shown under them up to date. With **Auto-update** ticked the cleaning re-runs when a value changes (sliders apply when released); otherwise press **Run cleaning**.

The farm here is independent of the one chosen at the top of the notebook: switching it loads that farm's turbines, columns and fault windows for this section only, and picks its default wind speed and power columns.

Choices we made, since the CARE data differs from the paper's:

- **Power is normalised here** (it peaks near 1.0, see the power curves above), so `rated_power` defaults to 1.0 instead of the paper's kW.
- **`cap_tolerance`.** The paper's upper cut-off is 2500 kW, and its turbines' power clusters around 2000 kW above rated, so we default to 1.25 × rated.
- **`ar_power_fraction`.** The paper's above-rated cut-off is 1600 kW, so we default to 0.8 × rated.
- **`rated_wind_speed`.** The paper doesn't give one, so it's a slider. Check it against the power curve: it is drawn as a dashed line in the plots.
- **`cut_in_ws` and `drop_below_cut_in`** set our cut-in filter (step 1b). Defaults are 3 m/s and off.
- **`period`.** Each dataset is a training year followed by a prediction period. You can clean all rows, or one part only.

In [280]:
import pprint

style = {"description_width": "150px"}
wide = widgets.Layout(width="360px")

def _slider(cls, value, lo, hi, step, description, **kw):
    return cls(value=value, min=lo, max=hi, step=step, description=description, style=style, layout=wide,
               continuous_update=False, **kw)

# ---- Farm-specific metadata, loaded on demand so this section can use a different farm from the rest of the notebook.
_farm_ctx = {}

def farm_context(farm):
    """Event table, dataset -> turbine index, turbine IDs and feature options for one farm (cached)."""
    if farm not in _farm_ctx:
        files = sorted((DATA_DIR / farm / "datasets").glob("*.parquet"), key=lambda p: int(p.stem))
        ds = pd.DataFrame({
            "event_id": [int(p.stem) for p in files],
            "asset_id": [int(pq.read_table(p, columns=["asset_id"]).column(0)[0].as_py()) for p in files],
        })
        desc = load_feature_description(farm).set_index("sensor_name")["description"]
        feats = pd.DataFrame({"column": [c for c in pq.read_schema(files[0]).names if c not in META_COLUMNS]})
        feats["description"] = feats["column"].map(sensor_name).map(desc)
        _farm_ctx[farm] = {
            "event_info": load_event_info(farm).rename(columns={"asset": "asset_id"}),
            "datasets": ds,
            "turbine_ids": sorted(ds["asset_id"].unique().tolist()),
            "features": feats,
            "feature_options": [(c if pd.isna(d) else f"{c}  ({d})", c)
                                for c, d in zip(feats["column"], feats["description"])],
        }
    return _farm_ctx[farm]

def farm_default_col(ctx, prefix, prefer=None):
    """First `<prefix>*_avg` column of the farm, preferring one whose description contains `prefer`."""
    f = ctx["features"]
    cands = f[f["column"].str.startswith(prefix) & f["column"].str.endswith("_avg")]
    if prefer is not None:
        hit = cands[cands["description"].str.contains(prefer, case=False, na=False)]
        cands = hit if len(hit) else cands
    return cands["column"].iloc[0] if len(cands) else f["column"].iloc[0]

# While > 0, widget changes don't update CLEAN_CFG or re-run the cleaning. Used when several widgets change
# together (switching farm, filtering a column list), so the cleaning runs once on the finished config.
_cfg_batch = [0]

def _column_picker(description):
    """Column dropdown for the selected farm, with a filter box. Returns (widget, dropdown, filter)."""
    dd = widgets.Dropdown(description=description, style={"description_width": "80px"},
                          layout=widgets.Layout(width="270px"))
    filt = widgets.Text(placeholder="filter", layout=widgets.Layout(width="86px"))
    def on_filter(change):
        all_opts = farm_context(cfg_farm.value)["feature_options"]
        query, current = change["new"].lower(), dd.value
        opts = [o for o in all_opts if query in o[0].lower()]
        keep = [o for o in all_opts if o[1] == current]
        if keep and keep[0] not in opts:
            opts = keep + opts  # keep the current choice so it doesn't jump
        if not opts:
            return
        _cfg_batch[0] += 1
        try:
            dd.options = opts
            dd.value = current if keep else opts[0][1]
        finally:
            _cfg_batch[0] -= 1
    filt.observe(on_filter, names="value")
    return widgets.HBox([dd, filt]), dd, filt

cfg_farm = widgets.Dropdown(options=available_farms, value=FARM, description="Farm", style=style, layout=wide)
cfg_turbine = widgets.Dropdown(description="Turbine", style=style, layout=wide)
cfg_ws_widget, cfg_ws, cfg_ws_filter = _column_picker("Wind speed")
cfg_p_widget, cfg_p, cfg_p_filter = _column_picker("Power")

def apply_farm(farm):
    """Fill the turbine and column pickers for `farm` and select its default wind speed and power columns."""
    ctx = farm_context(farm)
    _cfg_batch[0] += 1
    try:
        cfg_turbine.options = [(f"Turbine {t}", t) for t in ctx["turbine_ids"]]
        cfg_turbine.value = ctx["turbine_ids"][0]
        for dd, filt, prefix, prefer in ((cfg_ws, cfg_ws_filter, "wind_speed", None),
                                         (cfg_p, cfg_p_filter, "power", "grid power")):
            filt.value = ""
            dd.options = ctx["feature_options"]
            dd.value = farm_default_col(ctx, prefix, prefer)
    finally:
        _cfg_batch[0] -= 1

apply_farm(cfg_farm.value)
# Registered before the CLEAN_CFG observers below, so the pickers are refilled before the config updates and the
# cleaning re-runs (once) for the new farm.
cfg_farm.observe(lambda change: apply_farm(change["new"]), names="value")
cfg_period = widgets.Dropdown(options=[("All rows", "all"), ("Train only", "train"), ("Prediction only", "prediction")],
                              description="Period", style=style, layout=wide)
cfg_rated_power = widgets.BoundedFloatText(value=1.0, min=1e-6, max=1e6, step=0.05, description="Rated power",
                                           style=style, layout=wide)
cfg_cap_tol = _slider(widgets.FloatSlider, 1.25, 1.0, 2.0, 0.05, "Cap tolerance (x rated)")
cfg_cut_in = _slider(widgets.FloatSlider, 3.0, 0.0, 6.0, 0.1, "Cut-in wind speed (m/s)")
cfg_drop_below = widgets.Checkbox(value=False, description="Also drop all records below cut-in", indent=False)
cfg_bin_width = _slider(widgets.FloatSlider, 1.0, 0.25, 3.0, 0.25, "Bin width (m/s)")
cfg_min_pts = _slider(widgets.IntSlider, 10, 3, 200, 1, "Min records per bin")
cfg_method = widgets.ToggleButtons(options=["mahalanobis", "euclidean"], description="Distance", style=style)
cfg_n_std = _slider(widgets.FloatSlider, 2.0, 0.5, 5.0, 0.1, "Threshold (std of distance)")
cfg_scope = widgets.ToggleButtons(options=[("Per bin", "bin"), ("All bins", "global")],
                                  description="Threshold computed", style=style)
cfg_ar = widgets.Checkbox(value=False, description="Above rated filter (AR)", indent=False)
cfg_rated_ws = _slider(widgets.FloatSlider, 12.0, 5.0, 20.0, 0.5, "Rated wind speed (m/s)")
cfg_ar_frac = _slider(widgets.FloatSlider, 0.8, 0.3, 1.0, 0.05, "AR min power (x rated)")
cfg_auto = widgets.Checkbox(value=True, description="Auto-update", indent=False)
cfg_view = widgets.HTML()

CFG_WIDGETS = {
    "farm": cfg_farm, "turbine": cfg_turbine, "ws_col": cfg_ws, "power_col": cfg_p, "period": cfg_period,
    "rated_power": cfg_rated_power, "cap_tolerance": cfg_cap_tol, "cut_in_ws": cfg_cut_in,
    "drop_below_cut_in": cfg_drop_below, "bin_width": cfg_bin_width,
    "min_bin_points": cfg_min_pts, "method": cfg_method, "n_std": cfg_n_std, "threshold_scope": cfg_scope,
    "above_rated_filter": cfg_ar, "rated_wind_speed": cfg_rated_ws, "ar_power_fraction": cfg_ar_frac,
}

def update_cfg(_=None):
    global CLEAN_CFG
    if _cfg_batch[0]:
        return
    CLEAN_CFG = {name: w.value for name, w in CFG_WIDGETS.items()}
    cfg_view.value = ('<pre style="font-size:11px; line-height:1.3">CLEAN_CFG = '
                      + pprint.pformat(CLEAN_CFG, sort_dicts=False, width=50) + "</pre>")

for w in CFG_WIDGETS.values():
    w.observe(update_cfg, names="value")
update_cfg()

clean_button = widgets.Button(description="Run cleaning", button_style="primary")
clean_out = widgets.Output(layout=widgets.Layout(flex="1 1 auto", min_width="0"))
clean_out.append_stdout("Results appear here once the last cell of this section has run.")

def _group(title, *children):
    return widgets.VBox([widgets.HTML(f"<b>{title}</b>"), *children])

cfg_panel = widgets.VBox([
    _group("Data", cfg_farm, cfg_turbine, cfg_ws_widget, cfg_p_widget, cfg_period),
    _group("Steps 1 and 1b: power and cut-in", cfg_rated_power, cfg_cap_tol, cfg_cut_in, cfg_drop_below),
    _group("Steps 2-5: distance filter", cfg_bin_width, cfg_min_pts, cfg_method, cfg_n_std, cfg_scope),
    _group("Above rated filter", cfg_ar, cfg_rated_ws, cfg_ar_frac),
    widgets.HBox([clean_button, cfg_auto]),
    cfg_view,
], layout=widgets.Layout(width="380px", flex="0 0 380px"))

display(widgets.HBox([cfg_panel, clean_out], layout=widgets.Layout(gap="16px", align_items="flex-start")))

### Step 0: Missing values (our addition)

This step is not in the paper. We remove records where wind speed or power is missing (NaN), or where **both are exactly 0**, which in SCADA data usually means the signal wasn't logged rather than a real reading. A record with zero power at a non-zero wind speed is a real observation (e.g. downtime), so it is left for the paper's steps to judge.

In [281]:
def missing_filter(df, cfg):
    """Step 0 (our addition, not in McKinnon et al.): flag records with missing wind speed or power, or both exactly 0."""
    ws, p = df[cfg["ws_col"]], df[cfg["power_col"]]
    return ws.isna() | p.isna() | ((ws == 0) & (p == 0))

### Step 1: Negative power filter

The paper: *"remove any data with either a negative power value, or a power value over 2500 kW (for simplicity this is referenced to as the negative power filter). These outliers are typically due to sensor error or turbine downtime."* In their Table 1 this step alone removed 10–18% of the raw data.

Our choice: the upper limit is `rated_power × cap_tolerance` (1.0 × 1.25 by default; see Configuration).

In [282]:
def power_filter(df, cfg):
    """Step 1, McKinnon et al. 'negative power filter': flag power < 0 or power > rated_power * cap_tolerance."""
    p = df[cfg["power_col"]]
    return (p < 0) | (p > cfg["rated_power"] * cfg["cap_tolerance"])

### Step 1b: Cut-in filter (our addition)

This step is not in the paper. Above cut-in wind speed, a power reading of 0 or less means the turbine isn't producing (downtime, service, a fault, or curtailment to zero). That isn't power-curve behaviour, and a long stoppage puts thousands of points at zero power in one bin, which drags the bin centre down and weakens the distance filter for that whole bin. So we remove records with **power ≤ 0 at wind speed ≥ `cut_in_ws`**.

The two farms log a stopped turbine differently. Farm A logs slightly negative power (removed by step 1). Farm C logs exactly 0, which step 1 keeps: that's 18.8% of its records, and about 113k of them are at 5 m/s or more. This step treats both farms the same.

Records below cut-in are kept by default, including real low-power production at the start of the curve. Optionally, **`drop_below_cut_in`** (off by default) removes every record below cut-in as well, labelled `below_cut_in` so it's distinguishable from stopped-turbine records. Use it if you only want the producing part of the curve, since idling below cut-in tells us little about faults.

In [283]:
def cut_in_filter(df, cfg):
    """Step 1b (our addition, not in McKinnon et al.): flag records with power <= 0 at wind speed >= cut_in_ws
    (turbine stopped while there is enough wind to produce)."""
    ws, p = df[cfg["ws_col"]], df[cfg["power_col"]]
    return (ws >= cfg["cut_in_ws"]) & (p <= 0)


def below_cut_in_filter(df, cfg):
    """Step 1b option (our addition): if drop_below_cut_in, flag every record with wind speed < cut_in_ws."""
    if not cfg["drop_below_cut_in"]:
        return pd.Series(False, index=df.index)
    return df[cfg["ws_col"]] < cfg["cut_in_ws"]

### Step 2: Binning

The paper: *"data bins of roughly 1 m/s width were constructed."*

Our choice: the bins are fixed-width (`bin_width`, default 1 m/s) and start at 0 m/s, so bin *k* covers [*k*·w, (*k*+1)·w).

In [284]:
def bin_wind_speed(df, cfg):
    """Step 2, McKinnon et al. binning: wind-speed bin index of each record (bins of bin_width m/s, from 0 m/s)."""
    return np.floor(df[cfg["ws_col"]] / cfg["bin_width"]).astype(int)

### Step 3: Bin centre

The paper: *"The mean power and wind speed values of these bins were found as cluster centres."*

The paper doesn't say whether the covariance matrix in step 4 is taken over the whole dataset or per bin. Its explanation suggests per bin: the Mahalanobis distance *"takes into account the angle of the distribution below rated"*, and data in each bin are *"distributed in truncated ellipses"*. So we compute the covariance per bin, alongside the centre. Bins with fewer than `min_bin_points` records get no centre, and their records are kept.

In [285]:
import numpy as np


def bin_centres(df, bins, cfg):
    """Step 3, McKinnon et al. bin centre: per bin, mean wind speed and mean power (+ the bin's covariance for step 4)."""
    xy = df[[cfg["ws_col"], cfg["power_col"]]]
    rows = {}
    for b, g in xy.groupby(bins):
        values = g.to_numpy(dtype=float)
        rows[b] = {
            "ws_low": b * cfg["bin_width"],
            "n": len(g),
            "ws_mean": values[:, 0].mean(),
            "power_mean": values[:, 1].mean(),
            "cov": np.cov(values, rowvar=False) if len(g) > 1 else np.full((2, 2), np.nan),
        }
    return pd.DataFrame.from_dict(rows, orient="index").rename_axis("bin")

### Step 4: Mahalanobis distance

The paper (equation 1): $D_{\mathrm{Mahalanobis}}(x, y) = \sqrt{(x - y)\,C^{-1}(x - y)}$, where for each record $x = (\text{wind speed}, \text{power})$, $y = \mu_{\text{bin}}$ is the bin centre and $C$ the covariance matrix. Mahalanobis *"takes into account the covariance of the data"*, whereas the Euclidean distance the authors compared against (equation 2) *"assumes the data is distributed in a circle"*.

Our choices:
- We use `scipy.spatial.distance.mahalanobis` with the bin's inverse covariance. We use a pseudo-inverse, so a bin where power never changes (singular covariance) doesn't crash.
- `method="euclidean"` reproduces the paper's comparison method. Note that it depends on units. Here power is normalised (0–1) while wind speed is in m/s, so wind speed dominates the Euclidean distance; in the paper, power in kW would have dominated instead.

In [286]:
from scipy.spatial.distance import euclidean, mahalanobis


def point_distances(df, bins, centres, cfg, method="mahalanobis"):
    """Step 4, McKinnon et al. eq. 1 (Mahalanobis) or eq. 2 (Euclidean): distance of each record from its bin centre.

    NaN for records in bins with fewer than cfg["min_bin_points"] records.
    """
    xy = df[[cfg["ws_col"], cfg["power_col"]]].to_numpy(dtype=float)
    bin_of = bins.to_numpy()
    dist = np.full(len(df), np.nan)
    for b, c in centres.iterrows():
        if c["n"] < cfg["min_bin_points"]:
            continue
        idx = np.flatnonzero(bin_of == b)
        mu = np.array([c["ws_mean"], c["power_mean"]])
        if method == "mahalanobis":
            VI = np.linalg.pinv(c["cov"])
            dist[idx] = [mahalanobis(xy[i], mu, VI) for i in idx]
        elif method == "euclidean":
            dist[idx] = [euclidean(xy[i], mu) for i in idx]
        else:
            raise ValueError(f"unknown method {method!r}")
    return pd.Series(dist, index=df.index)

### Step 5: Threshold

The paper: *"a threshold of 2 standard deviations of the distance was used for both the Mahalanobis and Euclidean distances. This, in theory, should remove only around 5% of the data."* In their Table 1 the Mahalanobis method removed **5.1–6.6%** of the power-filtered data (5.2–7.5% with the above-rated filter).

The paper doesn't define the threshold more precisely. We remove records whose distance is above **mean + `n_std` × standard deviation** of the distances. By default the mean and standard deviation are computed per bin (`threshold_scope="bin"`), matching the per-bin centres; `"global"` uses one threshold over all bins. `distance_filter` runs steps 2–5 together.

In [287]:
def distance_filter(df, cfg, method="mahalanobis"):
    """Steps 2-5, McKinnon et al.: bin, find bin centres, measure distances, and flag records whose distance is
    above mean + n_std * std of the distance ("a threshold of 2 standard deviations of the distance").

    Returns (remove_mask, info) where info holds bins, distances, thresholds and the per-bin centre table.
    """
    bins = bin_wind_speed(df, cfg)
    centres = bin_centres(df, bins, cfg)
    dist = point_distances(df, bins, centres, cfg, method)
    if cfg["threshold_scope"] == "bin":
        stats = dist.groupby(bins).agg(["mean", "std"])
        bin_threshold = stats["mean"] + cfg["n_std"] * stats["std"]
    elif cfg["threshold_scope"] == "global":
        bin_threshold = pd.Series(dist.mean() + cfg["n_std"] * dist.std(), index=centres.index)
    else:
        raise ValueError(f"unknown threshold_scope {cfg['threshold_scope']!r}")
    threshold = bins.map(bin_threshold)
    remove = dist > threshold  # NaN distance (too-small bin) compares False, so those records are kept
    centres["threshold"] = bin_threshold
    centres["removed"] = remove.groupby(bins).sum()
    return remove, {"bins": bins, "distance": dist, "threshold": threshold, "centres": centres}

### Above rated filter (optional)

The paper also tested a variant ("AR") in which the distance filter is applied **only below rated wind speed**. Above rated, *"a general filter of any power value lower than 1600 kW was removed"*, because the data there were *"sparse"* and closer to a Poisson distribution than a Gaussian one, so a standard-deviation threshold doesn't suit them. They found the AR filter removes curtailment lines just below rated power that the distance filter misses.

Our choice: above rated means wind speed ≥ `rated_wind_speed`, and the power cut-off is `ar_power_fraction × rated_power` (0.8 × 1.0 by default, from 1600/2000 kW). The filter is off by default, so the default run matches the paper's main method.

In [288]:
def above_rated_filter(df, cfg):
    """McKinnon et al. 'above rated filter' (AR): at wind speed >= rated, flag power < ar_power_fraction * rated_power."""
    ws, p = df[cfg["ws_col"]], df[cfg["power_col"]]
    return (ws >= cfg["rated_wind_speed"]) & (p < cfg["ar_power_fraction"] * cfg["rated_power"])

### Full pipeline

`mckinnon_clean` applies the steps in order, each to the records the previous steps kept. With the AR filter on, the distance filter only sees records below rated wind speed, and `above_rated_filter` handles the rest. It returns:

- `df_flagged`: the input with a fresh index plus `removed_by` (`kept`, `missing`, `power_filter`, `cut_in_filter`, `below_cut_in`, `distance_filter` or `above_rated_filter`), `bin`, `distance` and `threshold`. The cleaned data is `df_flagged[df_flagged["removed_by"] == "kept"]`.
- `params`: the config used and the per-bin table (centre, covariance, threshold, records removed).

In [289]:
REMOVAL_STEPS = ["missing", "power_filter", "cut_in_filter", "below_cut_in", "distance_filter", "above_rated_filter"]


def mckinnon_clean(df, cfg):
    """McKinnon et al. (2022) cleaning: step 0 (missing, ours), step 1 (negative power filter), step 1b
    (cut-in, ours), steps 2-5 (binned distance filter), optional above-rated filter. Returns (df_flagged, params)."""
    out = df.reset_index(drop=True).copy()
    out["removed_by"] = "kept"

    def flag(mask, label):
        out.loc[mask[mask].index, "removed_by"] = label

    flag(missing_filter(out, cfg), "missing")
    flag(power_filter(out[out["removed_by"] == "kept"], cfg), "power_filter")
    flag(cut_in_filter(out[out["removed_by"] == "kept"], cfg), "cut_in_filter")
    flag(below_cut_in_filter(out[out["removed_by"] == "kept"], cfg), "below_cut_in")

    remaining = out[out["removed_by"] == "kept"]
    if cfg["above_rated_filter"]:
        flag(above_rated_filter(remaining, cfg), "above_rated_filter")
        remaining = remaining[remaining[cfg["ws_col"]] < cfg["rated_wind_speed"]]
    remove, info = distance_filter(remaining, cfg, cfg["method"])
    flag(remove, "distance_filter")

    out["bin"] = info["bins"]
    out["distance"] = info["distance"]
    out["threshold"] = info["threshold"]
    out["removed_by"] = pd.Categorical(out["removed_by"], categories=["kept", *REMOVAL_STEPS])
    return out, {"cfg": dict(cfg), "centres": info["centres"]}

### Run and compare before / after

The table counts each step's removals. **% of filtered** is the share of the records that reach the distance filter (after steps 0, 1 and 1b). It matches the paper's Table 1 measure (5.1–6.6% for the Mahalanobis method), because the paper's base was the data left after its only pre-filter, step 1. **Fault-window records removed** shows how much of the labelled fault data the cleaning throws away. This matters for fault detection, because cleaning that removes the off-curve behaviour may also remove the fault signature.

The figure shows:
- **Before:** every record, coloured by the step that removed it.
- **After:** the kept records, with the bin centres.
- **Below:** the share of each bin removed by the distance filter.

The last run is kept in `cleaned_flagged` and `cleaned_params`.

In [290]:
CLEAN_COLORS = {"kept": "#b5b4ad", "missing": "#2a78d6", "power_filter": "#eb6834", "cut_in_filter": "#e87ba4",
                "below_cut_in": "#6250d6", "distance_filter": "#1baf7a", "above_rated_filter": "#eda100"}
_clean_cache = {}


def turbine_power_data(farm, asset_id, ws_col, power_col, period):
    """One turbine's rows (wind speed, power, status, `in_fault` for any labelled fault window), cached.

    Like load_turbine_rows above, but for any farm rather than the notebook's FARM.
    """
    key = (farm, asset_id, ws_col, power_col)
    if key not in _clean_cache:
        ctx = farm_context(farm)
        ds, ev = ctx["datasets"], ctx["event_info"]
        cols = list(dict.fromkeys([ws_col, power_col]))
        frames = [load_event(farm, eid, columns=cols) for eid in ds.loc[ds["asset_id"] == asset_id, "event_id"]]
        df = pd.concat(frames).drop_duplicates("time_stamp").sort_values("time_stamp")
        df["in_fault"] = False
        faults = ev[(ev["asset_id"] == asset_id) & (ev["event_label"] == "anomaly")]
        for start, end in zip(faults["event_start"], faults["event_end"]):
            df.loc[df["time_stamp"].between(start, end), "in_fault"] = True
        _clean_cache[key] = df
    df = _clean_cache[key]
    return df if period == "all" else df[df["train_test"] == period]


def cleaning_stats(flagged):
    counts = flagged["removed_by"].value_counts()
    n_total = len(flagged)
    n_filtered = n_total - counts[["missing", "power_filter", "cut_in_filter", "below_cut_in"]].sum()
    rows = []
    for step in REMOVAL_STEPS:
        n = counts[step]
        late_step = step in ("distance_filter", "above_rated_filter")
        rows.append({"step": step, "records removed": n, "% of original": 100 * n / n_total,
                     "% of filtered": 100 * n / n_filtered if late_step else np.nan})
    n_dist = counts["distance_filter"] + counts["above_rated_filter"]
    rows.append({"step": "distance + AR (paper's Table 1 measure)", "records removed": n_dist,
                 "% of original": 100 * n_dist / n_total, "% of filtered": 100 * n_dist / n_filtered})
    n_removed = n_total - counts["kept"]
    rows.append({"step": "total removed", "records removed": n_removed, "% of original": 100 * n_removed / n_total,
                 "% of filtered": np.nan})
    in_fault = flagged["in_fault"]
    if in_fault.any():
        n_fault_removed = (in_fault & (flagged["removed_by"] != "kept")).sum()
        rows.append({"step": "fault-window records removed", "records removed": n_fault_removed,
                     "% of original": np.nan, "% of filtered": np.nan,
                     "% of fault-window records": 100 * n_fault_removed / in_fault.sum()})
    table = pd.DataFrame(rows).set_index("step").reindex(
        columns=["records removed", "% of original", "% of filtered", "% of fault-window records"])
    return (table.style
            .format({"records removed": "{:,}", "% of original": "{:.2f}", "% of filtered": "{:.2f}",
                     "% of fault-window records": "{:.2f}"}, na_rep="")
            .set_caption(f"{n_total:,} records in, {counts['kept']:,} kept"))


def _style_axes(ax):
    ax.grid(True, color="#e6e5e0", lw=0.6)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color("#b5b4ad")
    ax.tick_params(colors="#52514e", labelsize=9)


def plot_cleaning(flagged, params, cfg):
    ws_col, p_col = cfg["ws_col"], cfg["power_col"]
    centres = params["centres"]
    counts = flagged["removed_by"].value_counts()
    kept = flagged[flagged["removed_by"] == "kept"]
    pct_removed = 100 * (1 - len(kept) / len(flagged))

    fig = plt.figure(figsize=(10.5, 8))
    gs = fig.add_gridspec(2, 2, height_ratios=[3, 1.2], hspace=0.35)
    ax_before = fig.add_subplot(gs[0, 0])
    ax_after = fig.add_subplot(gs[0, 1], sharex=ax_before, sharey=ax_before)
    ax_bins = fig.add_subplot(gs[1, :])

    for step in ["kept", *REMOVAL_STEPS]:
        pts = flagged[flagged["removed_by"] == step]
        if len(pts):
            ax_before.scatter(pts[ws_col], pts[p_col], s=3, alpha=0.35 if step == "kept" else 0.6,
                              color=CLEAN_COLORS[step], lw=0, rasterized=True)
        ax_before.scatter([], [], s=30, color=CLEAN_COLORS[step], label=f"{step} ({counts[step]:,})")
    ax_before.set_title(f"Before: {len(flagged):,} records", loc="left", fontsize=10, color="#0b0b0b")
    ax_before.legend(loc="upper left", frameon=False, fontsize=8)

    ax_after.scatter(kept[ws_col], kept[p_col], s=3, alpha=0.35, color="#2a78d6", lw=0, rasterized=True)
    used = centres[centres["n"] >= cfg["min_bin_points"]]
    ax_after.plot(used["ws_mean"], used["power_mean"], "o-", color="#0b0b0b", ms=4, lw=1.2, label="Bin centres")
    ax_after.set_title(f"After: {len(kept):,} records ({pct_removed:.1f}% removed)", loc="left", fontsize=10,
                       color="#0b0b0b")
    ax_after.legend(loc="upper left", frameon=False, fontsize=9)

    for ax in (ax_before, ax_after):
        if cfg["above_rated_filter"]:
            ax.axvline(cfg["rated_wind_speed"], color="#52514e", lw=1, ls="--")
            ax.axhline(cfg["ar_power_fraction"] * cfg["rated_power"], color="#52514e", lw=0.8, ls=":")
        ax.set_xlabel(f"{ws_col} [m/s]", color="#52514e")
        _style_axes(ax)
    ax_before.set_ylabel(p_col, color="#52514e")
    for ax in (ax_before, ax_after):  # cut-in wind speed (step 1b)
        ax.axvline(cfg["cut_in_ws"], color="#e87ba4", lw=1, ls="--")

    filtered_bins = centres[centres["n"] >= cfg["min_bin_points"]]  # smaller bins aren't distance-filtered
    share = 100 * filtered_bins["removed"] / filtered_bins["n"]
    ax_bins.bar(filtered_bins["ws_low"] + cfg["bin_width"] / 2, share, width=cfg["bin_width"] * 0.9,
                color="#1baf7a", edgecolor="none")
    ax_bins.axhline(5, color="#52514e", lw=0.8, ls=":")
    ax_bins.text(ax_bins.get_xlim()[1], 5, " ~5% expected", va="center", ha="left", fontsize=8, color="#52514e")
    ax_bins.set_title(f"Removed by the {cfg['method']} distance filter, per {cfg['bin_width']:g} m/s bin",
                      loc="left", fontsize=10, color="#0b0b0b")
    ax_bins.set_xlabel("Wind speed bin [m/s]", color="#52514e")
    ax_bins.set_ylabel("% of bin", color="#52514e")
    _style_axes(ax_bins)

    fig.suptitle(f"Farm {cfg['farm']}, turbine {cfg['turbine']}: McKinnon et al. cleaning ({cfg['method']}, "
                 f"{cfg['n_std']:g} std, {'AR on' if cfg['above_rated_filter'] else 'AR off'})",
                 x=0.07, ha="left", fontsize=11, color="#0b0b0b")
    plt.close(fig)
    return fig


def make_cleaning():
    global cleaned_flagged, cleaned_params
    cfg = dict(CLEAN_CFG)
    df = turbine_power_data(cfg["farm"], cfg["turbine"], cfg["ws_col"], cfg["power_col"], cfg["period"])
    if df.empty:
        raise ValueError("No rows for this turbine and period.")
    cleaned_flagged, cleaned_params = mckinnon_clean(df, cfg)
    return cleaning_stats(cleaned_flagged), plot_cleaning(cleaned_flagged, cleaned_params, cfg)


# Detach handlers from any earlier run of this cell. The settings widgets come from the configuration cell, so
# they outlive re-runs of this cell, and each run would otherwise add another handler (one extra render per run).
if "_on_clean_click" in globals():
    clean_button.on_click(_on_clean_click, remove=True)
    for w in CFG_WIDGETS.values():
        try:
            w.unobserve(_auto_rerun, names="value")
        except ValueError:  # widget recreated by re-running the configuration cell, so never had the handler
            pass


def _on_clean_click(_):
    render_plot(clean_out, clean_button, make_cleaning)


def _auto_rerun(_):
    if cfg_auto.value and not _cfg_batch[0]:
        _render_plot(clean_out, clean_button, make_cleaning)  # bypass the duplicate-click guard for value changes


clean_button.on_click(_on_clean_click)
for w in CFG_WIDGETS.values():
    w.observe(_auto_rerun, names="value")

_render_plot(clean_out, clean_button, make_cleaning)  # results go in the panel beside the settings above